# Phase 4: Court Judgment Prediction: GQA decoder

This is the **Court Judgment Prediction** part of the full `gqa` notebook (`notebooks/CS781_gqa_all_phases.ipynb`), extracted so it can be read on its own. It reuses the tokenizer, model classes and helpers defined in the earlier cells, so **to run it, execute the full notebook from the top** (or run the Phase 1 set-up cells first, then this part).

## Court Judgement Prediction

In [ ]:
import json, os, numpy as np
from huggingface_hub import snapshot_download

P4_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_4"
os.makedirs(P4_DIR, exist_ok=True)

# Downloads only the Phase-4 folder from the course dataset repo (same repo as Phase 3).
# If the folder name differs, change allow_patterns.
local = snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                          allow_patterns=["Phase_4_CJPE/*", "*/Phase_4_CJPE/*"],
                          token=HF_TOKEN)
print("downloaded to:", local)
for r, _, fs in os.walk(local):
    for f in fs:
        if f.endswith(".json"): print(os.path.join(r, f))

def find(name):
    for r, _, fs in os.walk(local):
        if name in fs and "Phase_4" in r: return os.path.join(r, name)

def load(name):
    d = json.load(open(find(name)))
    print(name, type(d).__name__, len(d))
    return d

tr, dv, te = load("train.json"), load("dev.json"), load("test.json")

def peek(d):
    x = d[0] if isinstance(d, list) else {k: (v[0] if isinstance(v, list) else v) for k, v in d.items()}
    print({k: (str(v)[:80] + "…" if len(str(v)) > 80 else v) for k, v in x.items()})
for n, d in [("train", tr), ("dev", dv), ("test", te)]:
    print(n); peek(d)

# label balance + token lengths (uses your Phase-1 tokenizer from the setup cell)
def rows(d):
    return d if isinstance(d, list) else [dict(zip(d.keys(), v)) for v in zip(*d.values())]
for n, d in [("train", tr), ("dev", dv)]:
    R = rows(d)
    labs = np.array([r["label"] for r in R])
    lens = np.array([len(tokenizer.encode(r["text"]).ids) for r in R[:300]])
    print(n, "label counts:", np.bincount(labs), "| tokens (first 300 docs) median/p90/max:",
          int(np.median(lens)), int(np.percentile(lens, 90)), int(lens.max()))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

downloaded to: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Test/test.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/train.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/dev.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_2_LNER/Test/fold3-text.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_2_LNER/Train/fold1-text.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be7

In [ ]:
import pickle, time, numpy as np, os

CACHE = f"{P4_DIR}/cjpe_tok_cache.pkl"
HEAD_N = TAIL_N = 4096

def rows_of(d): return d
def tok_split(texts):
    out = []
    for i in range(0, len(texts), 500):
        batch = texts[i:i+500]
        # only the first/last 40k chars can reach the first/last 4096 tokens
        h = [t[:40000] for t in batch]
        tl = [t[-40000:] if len(t) > 80000 else None for t in batch]
        eh = tokenizer.encode_batch(h)
        et = tokenizer.encode_batch([x for x in tl if x is not None]) if any(x is not None for x in tl) else []
        k = 0
        for j, t in enumerate(batch):
            hi = eh[j].ids
            if tl[j] is None:                      # short enough: whole doc encoded
                ti, total = hi, len(hi)
            else:
                ti, total = et[k].ids, -1; k += 1  # total unknown (very long)
            out.append((np.array(hi[:HEAD_N], dtype=np.uint16),
                        np.array(ti[-TAIL_N:], dtype=np.uint16),
                        total if total >= 0 else 10**6))
    return out

if os.path.exists(CACHE):
    C = pickle.load(open(CACHE, "rb")); print("loaded cache")
else:
    t0 = time.time(); C = {}
    for name, d in [("train", tr), ("dev", dv), ("test", te)]:
        texts = [r["text"] for r in d]
        toks = tok_split(texts)
        C[name] = dict(ids=[r["id"] for r in d],
                       y=(np.array([r["label"] for r in d]) if name != "test" else None),
                       head=[a for a, _, _ in toks], tail=[b for _, b, _ in toks],
                       n=np.array([n for _, _, n in toks]))
        print(name, len(d), f"{(time.time()-t0)/60:.1f} min")
    pickle.dump(C, open(CACHE, "wb"))

# sanity: test 'label' field must be ignored; just check if it is a constant placeholder
print("test label field values:", np.unique([r.get("label") for r in te], return_counts=True))
print("train docs ≤4096 tok:", (C["train"]["n"] <= 4096).mean().round(3),
      "| ≤8192:", (C["train"]["n"] <= 8192).mean().round(3))
print("✅ cache ready:", {k: len(v["ids"]) for k, v in C.items()})

train 37387 1.5 min
dev 3505 1.7 min
test 1517 1.7 min
test label field values: (array([0, 1]), array([755, 762]))
train docs ≤4096 tok: 0.694 | ≤8192: 0.948
✅ cache ready: {'train': 37387, 'dev': 3505, 'test': 1517}


In [ ]:
def build_gqa_rr(dropout=0.1):
    lm_ = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                          n_kv_head=4, max_seq_len=2048, tie_weights=True)
    ck = torch.load(TEACHER_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck["model"] if "model" in ck else ck)      # strict: pretrained Phase-1 weights
    return RRModel(lm_, 768, NC, dropout).to(device)


def cj_from(rr):
    """Phase-3 RRModel -> CJPE model: keep the Phase-1 LM, discard the old head, fresh 2-class head."""
    d = rr.lm.token_embedding.weight.shape[1]          # model width, independent of the old head
    m = CJModel(rr.lm, d, 2, 0.1).to(device)
    m.head = nn.Linear(2 * d, 2).to(device)
    nn.init.normal_(m.head.weight, std=0.02); nn.init.zeros_(m.head.bias)
    return m

In [ ]:
import json, time, math, os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

CH = 1024
MODES = {"first1": 1, "last1": 1, "first4": 4, "last4": 4, "ht4": 4}   # value = max chunks per doc

# ---------- document -> chunks (from the cached head/tail tokens) ----------
def get_chunks(split, i, mode):
    d = C[split]; n = int(d["n"][i]); head, tail = d["head"][i], d["tail"][i]
    if   mode == "first1": x = head[:CH]
    elif mode == "last1":  x = tail[-CH:]
    elif mode == "first4": x = head[:4*CH]
    elif mode == "last4":  x = tail[-4*CH:]
    elif mode == "ht4":    x = head if n <= 4*CH else np.concatenate([head[:2*CH], tail[-2*CH:]])
    L = len(x)
    if mode.startswith("last"):                      # align chunks to the END of the judgment
        ends = list(range(L, 0, -CH))[::-1]
        return [x[max(0, e-CH):e] for e in ends]
    return [x[s:s+CH] for s in range(0, L, CH)]

def collate_cj(split, idxs, mode):
    seqs, owner = [], []
    for j, i in enumerate(idxs):
        cs = get_chunks(split, i, mode)
        if not cs or len(cs[0]) == 0: cs = [np.zeros(1, np.uint16)]
        for c in cs: seqs.append(c); owner.append(j)
    T = max(len(s) for s in seqs)
    ids = torch.zeros(len(seqs), T, dtype=torch.long)
    for k, s in enumerate(seqs): ids[k, :len(s)] = torch.from_numpy(s.astype(np.int64))
    lens = torch.tensor([len(s) for s in seqs])
    return ids.to(device), lens.to(device), torch.tensor(owner, device=device), len(idxs)

# ---------- model: Phase-1 LM -> chunk features -> mean over chunks -> 2-class head ----------
class CJModel(RRModel):
    def forward(self, ids, lens, owner, n_docs):
        H = self.hidden(ids)                                            # (chunks, T, d)
        B, T, _ = H.shape
        mask = (torch.arange(T, device=H.device)[None] < lens[:, None]).unsqueeze(-1)
        mean = (H.float() * mask).sum(1) / lens[:, None]
        last = H[torch.arange(B, device=H.device), lens - 1].float()
        f = torch.cat([last, mean], -1)                                 # (chunks, 2d)
        D = torch.zeros(n_docs, f.size(1), device=f.device).index_add_(0, owner, f)
        D = D / torch.bincount(owner, minlength=n_docs).clamp(min=1)[:, None]
        return self.head(self.drop(D))

def cj_from(rr):
    """Turn a Phase-3 RRModel (13-label head) into a CJPE model: same LM, fresh 2-class head."""
    d = rr.head.in_features // 2
    return CJModel(rr.lm, d, 2, 0.1).to(device)

# ---------- inference / metrics ----------
@torch.no_grad()
def cj_logits(model, split, mode, idxs=None, chunk_batch=32):
    model.eval()
    idxs = np.arange(len(C[split]["ids"])) if idxs is None else np.asarray(idxs)
    bs = max(1, chunk_batch // MODES[mode]); out = np.zeros((len(idxs), 2), np.float32)
    for s in range(0, len(idxs), bs):
        b = idxs[s:s+bs]
        with torch.autocast("cuda", dtype=torch.bfloat16):
            lg = model(*collate_cj(split, b, mode))
        out[s:s+len(b)] = lg.float().cpu().numpy()
    return out

def cj_scores(y, lg, thr=0.5):
    p1 = torch.softmax(torch.tensor(lg), -1)[:, 1].numpy(); pr = (p1 > thr).astype(int)
    P, R, Fm, _ = precision_recall_fscore_support(y, pr, average="macro", zero_division=0)
    return dict(f1=Fm, p=P, r=R, cm=confusion_matrix(y, pr, labels=[0, 1]))

def best_thr(y, lg):
    best = (0.5, -1.0)
    for t in np.arange(0.2, 0.81, 0.02):
        f = cj_scores(y, lg, t)["f1"]
        if f > best[1]: best = (float(t), f)
    return best

# ---------- training (CE, or KD from cached teacher logits) ----------
def train_cj(model, name, mode, epochs=2, lr=5e-5, head_lr=1e-3, wd=0.01, bs_chunks=16,
             n_train=None, kd=None, alpha=0.5, T=2.0, evals_per_epoch=2, patience=3,
             class_weight=True, debug=False, seed=0):
    torch.manual_seed(seed); rng = np.random.RandomState(seed)
    ytr = C["train"]["y"]; ydv = C["dev"]["y"]
    dev_idx = np.arange(200) if debug else np.arange(len(ydv))
    tr_idx = rng.permutation(len(ytr))[: (128 if debug else (n_train or len(ytr)))]
    if debug: epochs, evals_per_epoch = 1, 1
    cw = None
    if class_weight:
        cnt = np.bincount(ytr, minlength=2); cw = torch.tensor(len(ytr) / (2 * cnt), dtype=torch.float, device=device)
    kd_t = torch.tensor(kd, dtype=torch.float) if kd is not None else None

    head_p = [p for p in model.head.parameters()]
    hid = {id(p) for p in head_p}
    body_p = [p for p in model.parameters() if p.requires_grad and id(p) not in hid]
    opt = torch.optim.AdamW([{"params": body_p, "lr": lr}, {"params": head_p, "lr": head_lr}], weight_decay=wd)
    bs = max(1, bs_chunks // MODES[mode]); spe = math.ceil(len(tr_idx) / bs); total = spe * epochs
    warm = max(1, int(0.06 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / (total - warm)))
    every = max(1, spe // evals_per_epoch)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad); n_all = sum(p.numel() for p in model.parameters())
    print(f"[{name}] mode={mode} train_docs={len(tr_idx)} steps={total} trainable={n_tr/1e6:.2f}M / {n_all/1e6:.1f}M")

    path = f"{P4_DIR}/cj_{name}_best.pt"; best = (-1.0, None); bad = 0; step = 0; t0 = time.time()
    torch.cuda.reset_peak_memory_stats(); hist = []; done = False
    for ep in range(epochs):
        perm = rng.permutation(tr_idx); model.train()
        for s in range(0, len(perm), bs):
            b = perm[s:s+bs]
            with torch.autocast("cuda", dtype=torch.bfloat16):
                lg = model(*collate_cj("train", b, mode)).float()
            y = torch.tensor(ytr[b], device=device)
            loss = F.cross_entropy(lg, y, weight=cw)
            if kd_t is not None:
                t_lg = kd_t[b].to(device)
                kl = F.kl_div(F.log_softmax(lg / T, -1), F.softmax(t_lg / T, -1), reduction="batchmean") * T * T
                loss = (1 - alpha) * loss + alpha * kl
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); step += 1
            if step == 20: print(f"  ~{(time.time()-t0)/20*total/60:.0f} min training total (excl. dev evals)")
            if step % 100 == 0: print(f"  step {step}/{total} loss {loss.item():.4f}")
            if step % every == 0 or step == total:
                lgd = cj_logits(model, "dev", mode, dev_idx); sc = cj_scores(ydv[dev_idx], lgd)
                hist.append((step, sc["f1"])); model.train()
                print(f"  eval@{step}: macro-F1 {sc['f1']:.4f}  P {sc['p']:.4f}  R {sc['r']:.4f}  ({(time.time()-t0)/60:.1f} min)")
                if sc["f1"] > best[0]:
                    best = (sc["f1"], step); bad = 0
                    torch.save({k: v.detach().cpu() for k, v in model.named_parameters() if v.requires_grad}, path)
                else:
                    bad += 1
                    if bad >= patience: done = True; break
        if done: break
    if os.path.exists(path):
        model.load_state_dict(torch.load(path), strict=False)
    lgd = cj_logits(model, "dev", mode, dev_idx); y = ydv[dev_idx]
    sc = cj_scores(y, lgd); thr, f_thr = best_thr(y, lgd)
    res = dict(name=name, mode=mode, f1=sc["f1"], p=sc["p"], r=sc["r"], cm=sc["cm"].tolist(), thr=thr, f1_thr=f_thr,
               minutes=(time.time()-t0)/60, peak_gb=torch.cuda.max_memory_allocated()/1e9,
               trainable=n_tr, total=n_all, best_step=best[1], dev_logits=lgd)
    print(f"✅ [{name}] dev macro-F1 {sc['f1']:.4f} (P {sc['p']:.4f} R {sc['r']:.4f}) | tuned thr {thr:.2f} → {f_thr:.4f} "
          f"| {res['minutes']:.1f} min, peak {res['peak_gb']:.1f} GB\n   confusion [[TN,FP],[FN,TP]] = {res['cm']}")
    return res

# ---------- outputs ----------
def write_pred(model, name, mode, thr=0.5):
    lg = cj_logits(model, "test", mode)
    p1 = torch.softmax(torch.tensor(lg), -1)[:, 1].numpy(); pred = (p1 > thr).astype(int)
    out = [{"id": i, "pred": int(p)} for i, p in zip(C["test"]["ids"], pred)]
    assert len(out) == len(C["test"]["ids"]) == 1517
    f = f"{P4_DIR}/prediction_{name}.json"; json.dump(out, open(f, "w"))
    np.save(f"{P4_DIR}/test_logits_{name}.npy", lg)
    print(f"wrote {f} | preds 0/1 = {np.bincount(pred, minlength=2).tolist()}  (rename to prediction.json when uploading)")

def save_kd_logits(model, name, mode):
    lg = cj_logits(model, "train", mode); np.save(f"{P4_DIR}/kd_logits_{name}.npy", lg); return lg

print("✅ Phase-4 toolkit ready")

✅ Phase-4 toolkit ready


In [ ]:
TEACHER_CKPT = f"{DRIVE_PROJECT_DIR}/gqa_checkpoints/gqa_best.pt"
NC = 13                       # only used to build the old Phase-3 head, which cj_from() then replaces

DEBUG   = False              # True = 128 train docs / 200 dev docs, ~1-2 min. Set False for the real run.
MODE    = "last4"             # last1 | first1 | first4 | last4 | ht4
N_TRAIN = None                # None = all 37k docs; set e.g. 15000 if the time estimate is too long
EPOCHS  = 2

rr = build_gqa_rr()
model = cj_from(rr); del rr
NAME = f"gqa_full_{MODE}"
CJ = globals().get("CJ", {})
CJ[NAME] = train_cj(model, NAME, MODE, epochs=EPOCHS, lr=5e-5, head_lr=1e-3,
                    n_train=N_TRAIN, debug=DEBUG)
if not DEBUG:
    write_pred(model, NAME, MODE)

[gqa_full_last4] mode=last4 train_docs=37387 steps=18694 trainable=87.51M / 87.5M
  ~50 min training total (excl. dev evals)
  step 100/18694 loss 0.9508
  step 200/18694 loss 0.4985
  step 300/18694 loss 0.6891
  step 400/18694 loss 0.7122
  step 500/18694 loss 1.6616
  step 600/18694 loss 1.7081
  step 700/18694 loss 1.6836
  step 800/18694 loss 0.9579
  step 900/18694 loss 0.4123
  step 1000/18694 loss 1.2417
  step 1100/18694 loss 1.6231
  step 1200/18694 loss 0.0806
  step 1300/18694 loss 0.1456
  step 1400/18694 loss 1.1627
  step 1500/18694 loss 1.0129
  step 1600/18694 loss 0.4453
  step 1700/18694 loss 0.0780
  step 1800/18694 loss 0.8932
  step 1900/18694 loss 0.9170
  step 2000/18694 loss 0.6645
  step 2100/18694 loss 0.8497
  step 2200/18694 loss 0.5085
  step 2300/18694 loss 2.6926
  step 2400/18694 loss 0.7140
  step 2500/18694 loss 0.1038
  step 2600/18694 loss 2.4826
  step 2700/18694 loss 3.3922
  step 2800/18694 loss 0.4142
  step 2900/18694 loss 0.1827
  step 3000/18

In [ ]:
t_lg = save_kd_logits(model, "gqa_full_last4", "last4")        # all 37k train docs, saved to Drive
print(t_lg.shape, "| teacher train accuracy:", (t_lg.argmax(1) == C["train"]["y"]).mean().round(4))

#   rr = build_gqa_rr(); model = cj_from(rr)
#   model.load_state_dict(torch.load(f"{P4_DIR}/cj_gqa_full_last4_best.pt"), strict=False)

(37387, 2) | teacher train accuracy: 0.9579


In [ ]:
STAGE1_CKPT = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_2/student_kd_stage1/best.pt"
STUDENT_CFG = dict(d_model=576, n_layer=7, n_head=12, n_kv_head=4)
def build_lm(cfg):
    return CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), max_seq_len=2048, tie_weights=True, **cfg)

def build_cj_student():
    lm_ = build_lm(STUDENT_CFG)
    ck = torch.load(STAGE1_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck["model"] if "model" in ck else ck)          # Phase-2 stage-1 student
    return cj_from(RRModel(lm_, STUDENT_CFG["d_model"], 13))

DEBUG   = False        # True first (~1-2 min), then False
MODE    = "last4"       # same view as the teacher
RUN_NOKD = False        # True also trains a no-teacher student baseline (costs another ~half hour)

kd_lg = np.load(f"{P4_DIR}/kd_logits_gqa_full_last4.npy")
assert kd_lg.shape == (len(C["train"]["ids"]), 2)

CJ = globals().get("CJ", {})
stu = build_cj_student()
print("student backbone params:", sum(p.numel() for p in stu.lm.parameters()) / 1e6, "M")
NAME = f"student_kd_{MODE}"
CJ[NAME] = train_cj(stu, NAME, MODE, epochs=2, lr=2e-4, head_lr=1e-3, kd=kd_lg, alpha=0.5, T=2.0, debug=DEBUG)
if not DEBUG:
    write_pred(stu, NAME, MODE)
del stu; torch.cuda.empty_cache()

if RUN_NOKD and not DEBUG:
    stu = build_cj_student(); NAME = f"student_nokd_{MODE}"
    CJ[NAME] = train_cj(stu, NAME, MODE, epochs=2, lr=2e-4, head_lr=1e-3, debug=False)
    del stu; torch.cuda.empty_cache()

student backbone params: 43.213248 M
[student_kd_last4] mode=last4 train_docs=37387 steps=18694 trainable=43.22M / 43.2M
  ~34 min training total (excl. dev evals)
  step 100/18694 loss 0.9877
  step 200/18694 loss 1.0478
  step 300/18694 loss 1.7748
  step 400/18694 loss 0.8892
  step 500/18694 loss 1.2192
  step 600/18694 loss 1.6822
  step 700/18694 loss 1.8313
  step 800/18694 loss 1.4654
  step 900/18694 loss 0.9772
  step 1000/18694 loss 0.7447
  step 1100/18694 loss 1.9169
  step 1200/18694 loss 1.3897
  step 1300/18694 loss 1.6111
  step 1400/18694 loss 1.3510
  step 1500/18694 loss 3.7161
  step 1600/18694 loss 2.2114
  step 1700/18694 loss 1.7126
  step 1800/18694 loss 1.2037
  step 1900/18694 loss 1.6970
  step 2000/18694 loss 0.5380
  step 2100/18694 loss 1.2732
  step 2200/18694 loss 1.2621
  step 2300/18694 loss 2.4057
  step 2400/18694 loss 1.2806
  step 2500/18694 loss 1.3678
  step 2600/18694 loss 1.6593
  step 2700/18694 loss 2.1572
  step 2800/18694 loss 1.9237
  ste

In [ ]:
# If the runtime restarted, reload the teacher first:
#   rr = build_gqa_rr(); model = cj_from(rr)
#   model.load_state_dict(torch.load(f"{P4_DIR}/cj_gqa_full_last4_best.pt"), strict=False)
ydv = C["dev"]["y"]
print(f"{'view':8s} {'F1':>7s} {'P':>7s} {'R':>7s}")
for m in ["first1", "last1", "first4", "ht4", "last4"]:
    lg = cj_logits(model, "dev", m); sc = cj_scores(ydv, lg)
    print(f"{m:8s} {sc['f1']:7.4f} {sc['p']:7.4f} {sc['r']:7.4f}")

view          F1       P       R
first1    0.5560  0.5561  0.5560
last1     0.6888  0.7045  0.6932
first4    0.6713  0.6732  0.6720
ht4       0.7115  0.7165  0.7128
last4     0.7170  0.7218  0.7182


In [ ]:
import glob, itertools, json, os, numpy as np, torch

# 1) save dev logits of everything trained in this session (the test logits were saved by write_pred)
for k, r in globals().get("CJ", {}).items():
    if len(r["dev_logits"]) == len(C["dev"]["y"]):                  # skip DEBUG runs (200 docs)
        np.save(f"{P4_DIR}/dev_logits_{k}.npy", r["dev_logits"])

# 2) load every model that has both dev and test logits
ydv = C["dev"]["y"]; D, Tt = {}, {}
for f in glob.glob(f"{P4_DIR}/dev_logits_*.npy"):
    k = os.path.basename(f)[len("dev_logits_"):-4]; tf = f"{P4_DIR}/test_logits_{k}.npy"
    if os.path.exists(tf) and len(np.load(f)) == len(ydv):
        D[k] = torch.softmax(torch.tensor(np.load(f)), -1).numpy()
        Tt[k] = torch.softmax(torch.tensor(np.load(tf)), -1).numpy()
print("models:", list(D))

# 3) every subset of size >= 2, equal-weight probability average, scored on dev
def dev_f1(p): return cj_scores(ydv, np.log(p + 1e-9))["f1"]
rows = [((k,), dev_f1(D[k])) for k in D]
for n in range(2, len(D) + 1):
    for sub in itertools.combinations(D, n):
        rows.append((sub, dev_f1(np.mean([D[k] for k in sub], 0))))
for sub, f in sorted(rows, key=lambda x: -x[1]): print(f"{f:.4f}  {' + '.join(sub)}")

# 4) write the best ensemble (only if it has >= 2 members)
best = max([r for r in rows if len(r[0]) >= 2], key=lambda x: x[1], default=None)
if best:
    sub, f = best
    p1 = np.mean([Tt[k] for k in sub], 0)[:, 1]; pred = (p1 > 0.5).astype(int)
    out = [{"id": i, "pred": int(p)} for i, p in zip(C["test"]["ids"], pred)]
    assert len(out) == 1517
    fn = f"{P4_DIR}/prediction_ens_best.json"; json.dump(out, open(fn, "w"))
    print(f"\nwrote {fn} | members: {sub} | dev macro-F1 {f:.4f}")

models: ['gqa_full_last4', 'student_kd_last4']
0.7252  gqa_full_last4 + student_kd_last4
0.7170  gqa_full_last4
0.5495  student_kd_last4

wrote /content/drive/MyDrive/CS781_Capstone_Project_Phase_4/prediction_ens_best.json | members: ('gqa_full_last4', 'student_kd_last4') | dev macro-F1 0.7252
